In [2]:
POSTS_PATH = "posts_sample.xml"
LANGUAGES_PATH = "programming-languages.csv"
OUTPUT_PATH = "output/top_languages_by_year_parquet"

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

Создаем SparkSession — основную точку входа для работы с Apache Spark.

In [ ]:
spark = (
    SparkSession.builder
    .appName("Lab2_Top_Programming_Languages")
    .master("local[*]")
    .config("spark.jars.packages", "com.databricks:spark-xml_2.12:0.18.0")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

Считываем XML-файл с постами Stack Overflow в Spark DataFrame.

In [19]:
posts_raw = (
    spark.read
    .format("xml")
    .option("rowTag", "row")
    .load(POSTS_PATH)
)

posts_raw.printSchema()

root
 |-- _AcceptedAnswerId: long (nullable = true)
 |-- _AnswerCount: long (nullable = true)
 |-- _Body: string (nullable = true)
 |-- _ClosedDate: timestamp (nullable = true)
 |-- _CommentCount: long (nullable = true)
 |-- _CommunityOwnedDate: timestamp (nullable = true)
 |-- _CreationDate: timestamp (nullable = true)
 |-- _FavoriteCount: long (nullable = true)
 |-- _Id: long (nullable = true)
 |-- _LastActivityDate: timestamp (nullable = true)
 |-- _LastEditDate: timestamp (nullable = true)
 |-- _LastEditorDisplayName: string (nullable = true)
 |-- _LastEditorUserId: long (nullable = true)
 |-- _OwnerDisplayName: string (nullable = true)
 |-- _OwnerUserId: long (nullable = true)
 |-- _ParentId: long (nullable = true)
 |-- _PostTypeId: long (nullable = true)
 |-- _Score: long (nullable = true)
 |-- _Tags: string (nullable = true)
 |-- _Title: string (nullable = true)
 |-- _ViewCount: long (nullable = true)



Считываем CSV-файл со списком языков программирования.

In [7]:
languages_raw = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", "\"")
    .csv(LANGUAGES_PATH)
)

languages_raw.printSchema()
languages_raw.show(10, truncate=False)
languages_raw.columns

root
 |-- name: string (nullable = true)
 |-- wikipedia_url: string (nullable = true)

+----------+---------------------------------------------------------+
|name      |wikipedia_url                                            |
+----------+---------------------------------------------------------+
|A# .NET   |https://en.wikipedia.org/wiki/A_Sharp_(.NET)             |
|A# (Axiom)|https://en.wikipedia.org/wiki/A_Sharp_(Axiom)            |
|A-0 System|https://en.wikipedia.org/wiki/A-0_System                 |
|A+        |https://en.wikipedia.org/wiki/A%2B_(programming_language)|
|A++       |https://en.wikipedia.org/wiki/A%2B%2B                    |
|ABAP      |https://en.wikipedia.org/wiki/ABAP                       |
|ABC       |https://en.wikipedia.org/wiki/ABC_(programming_language) |
|ABC ALGOL |https://en.wikipedia.org/wiki/ABC_ALGOL                  |
|ABSET     |https://en.wikipedia.org/wiki/ABSET                      |
|ABSYS     |https://en.wikipedia.org/wiki/ABSYS              

['name', 'wikipedia_url']

Подготавливаем список языков программирования для сравнения с тегами.

In [8]:
languages = (
    languages_raw
    .select(F.col("name").alias("language"))
    .filter(F.col("language").isNotNull())
    .withColumn("language", F.trim(F.col("language")))
    .withColumn("language_lower", F.lower(F.col("language")))
    .dropDuplicates(["language_lower"])
)

languages.show(20, truncate=False)
print("Количество языков:", languages.count())

+------------+--------------+
|language    |language_lower|
+------------+--------------+
|@Formula    |@formula      |
|A# (Axiom)  |a# (axiom)    |
|A# .NET     |a# .net       |
|A+          |a+            |
|A++         |a++           |
|A-0 System  |a-0 system    |
|ABAP        |abap          |
|ABC         |abc           |
|ABC ALGOL   |abc algol     |
|ABSET       |abset         |
|ABSYS       |absys         |
|ACC         |acc           |
|Accent      |accent        |
|Ace DASL    |ace dasl      |
|ACL2        |acl2          |
|ACT-III     |act-iii       |
|Action!     |action!       |
|ActionScript|actionscript  |
|Ada         |ada           |
|Adenine     |adenine       |
+------------+--------------+
only showing top 20 rows

Количество языков: 698


Выбираем из XML только те поля, которые понадобятся для анализа.

In [9]:
posts = (
    posts_raw
    .select(
        F.col("_Id").alias("Id"),
        F.col("_PostTypeId").alias("PostTypeId"),
        F.col("_CreationDate").alias("CreationDate"),
        F.col("_Tags").alias("Tags")
    )
)

posts.show(5, truncate=False)

+---+----------+-----------------------+------------------------------------------------------+
|Id |PostTypeId|CreationDate           |Tags                                                  |
+---+----------+-----------------------+------------------------------------------------------+
|4  |1         |2008-07-31 21:42:52.667|<c#><floating-point><type-conversion><double><decimal>|
|6  |1         |2008-07-31 22:08:08.62 |<html><css><internet-explorer-7>                      |
|7  |2         |2008-07-31 22:17:57.883|null                                                  |
|9  |1         |2008-07-31 23:40:59.743|<c#><.net><datetime>                                  |
|11 |1         |2008-07-31 23:55:37.967|<c#><datetime><time><datediff><relative-time-span>    |
+---+----------+-----------------------+------------------------------------------------------+
only showing top 5 rows



Оставляем только вопросы Stack Overflow и выбираем записи за период с 2010 по 2020 год.

In [10]:
questions = (
    posts
    .filter(F.col("PostTypeId") == "1")
    .withColumn("creation_ts", F.to_timestamp("CreationDate"))
    .withColumn("year", F.year("creation_ts"))
    .filter(F.col("year").between(2010, 2020))
    .filter(F.col("Tags").isNotNull())
)

questions.select("Id", "year", "Tags").show(10, truncate=False)

+-------+----+--------------------------------------+
|Id     |year|Tags                                  |
+-------+----+--------------------------------------+
|3768363|2010|<c++><character-encoding>             |
|3775996|2010|<sharepoint><infopath>                |
|3776721|2010|<iphone><app-store><in-app-purchase>  |
|3777993|2010|<symfony1><schema><doctrine><fixtures>|
|3778222|2010|<java>                                |
|3785460|2010|<visual-studio-2010><stylecop>        |
|3789116|2010|<cakephp><file-upload><swfupload>     |
|3794815|2010|<git><cygwin><putty>                  |
|3797876|2010|<drupal><drupal-6>                    |
|3798872|2010|<php><wordpress><memory>              |
+-------+----+--------------------------------------+
only showing top 10 rows



Разбираем строку с тегами и превращаем ее в отдельные значения.

In [11]:
questions_tags = (
    questions
    .withColumn("tags_clean", F.regexp_replace(F.col("Tags"), r"^<|>$", ""))
    .withColumn("tags_clean", F.regexp_replace(F.col("tags_clean"), r"><", ","))
    .withColumn("tag", F.explode(F.split(F.col("tags_clean"), ",")))
    .withColumn("tag", F.lower(F.trim(F.col("tag"))))
    .filter(F.col("tag") != "")
)

questions_tags.select("Id", "year", "tag").show(20, truncate=False)

+-------+----+------------------+
|Id     |year|tag               |
+-------+----+------------------+
|3768363|2010|c++               |
|3768363|2010|character-encoding|
|3775996|2010|sharepoint        |
|3775996|2010|infopath          |
|3776721|2010|iphone            |
|3776721|2010|app-store         |
|3776721|2010|in-app-purchase   |
|3777993|2010|symfony1          |
|3777993|2010|schema            |
|3777993|2010|doctrine          |
|3777993|2010|fixtures          |
|3778222|2010|java              |
|3785460|2010|visual-studio-2010|
|3785460|2010|stylecop          |
|3789116|2010|cakephp           |
|3789116|2010|file-upload       |
|3789116|2010|swfupload         |
|3794815|2010|git               |
|3794815|2010|cygwin            |
|3794815|2010|putty             |
+-------+----+------------------+
only showing top 20 rows



Сопоставляем теги вопросов со списком языков программирования.

In [12]:
language_mentions = (
    questions_tags
    .join(
        languages,
        questions_tags.tag == languages.language_lower,
        "inner"
    )
    .select(
        questions_tags.year,
        languages.language.alias("language")
    )
)

language_mentions.show(20, truncate=False)

+----+-----------+
|year|language   |
+----+-----------+
|2010|Java       |
|2010|PHP        |
|2010|Ruby       |
|2010|C          |
|2010|PHP        |
|2010|Python     |
|2010|JavaScript |
|2010|AppleScript|
|2010|PHP        |
|2010|PHP        |
|2010|JavaScript |
|2010|Sed        |
|2010|Python     |
|2010|Java       |
|2010|Ruby       |
|2010|Objective-C|
|2010|JavaScript |
|2010|R          |
|2010|PHP        |
|2010|JavaScript |
+----+-----------+
only showing top 20 rows



Подсчитываем, сколько раз каждый язык программирования встречается в каждом году.

In [13]:
year_language_counts = (
    language_mentions
    .groupBy("year", "language")
    .agg(F.count("*").alias("mentions"))
)

year_language_counts.orderBy("year", F.desc("mentions")).show(50, truncate=False)

+----+------------+--------+
|year|language    |mentions|
+----+------------+--------+
|2010|Java        |52      |
|2010|PHP         |46      |
|2010|JavaScript  |44      |
|2010|Python      |26      |
|2010|Objective-C |23      |
|2010|C           |20      |
|2010|Ruby        |12      |
|2010|Delphi      |8       |
|2010|Bash        |3       |
|2010|Perl        |3       |
|2010|R           |3       |
|2010|AppleScript |3       |
|2010|Sed         |2       |
|2010|Haskell     |2       |
|2010|F#          |2       |
|2010|ksh         |1       |
|2010|Scala       |1       |
|2010|PowerShell  |1       |
|2010|Mouse       |1       |
|2010|BASIC       |1       |
|2010|XPath       |1       |
|2010|dBase       |1       |
|2010|OCaml       |1       |
|2010|MATLAB      |1       |
|2010|Scheme      |1       |
|2010|Go          |1       |
|2010|ActionScript|1       |
|2010|Racket      |1       |
|2011|PHP         |102     |
|2011|Java        |93      |
|2011|JavaScript  |83      |
|2011|Python  

Формируем топ-10 языков для каждого года с помощью оконной функции.

In [14]:
window_spec = Window.partitionBy("year").orderBy(F.desc("mentions"), F.asc("language"))

top10_by_year = (
    year_language_counts
    .withColumn("rank", F.row_number().over(window_spec))
    .filter(F.col("rank") <= 10)
    .orderBy("year", "rank")
)

top10_by_year.show(200, truncate=False)

+----+-----------+--------+----+
|year|language   |mentions|rank|
+----+-----------+--------+----+
|2010|Java       |52      |1   |
|2010|PHP        |46      |2   |
|2010|JavaScript |44      |3   |
|2010|Python     |26      |4   |
|2010|Objective-C|23      |5   |
|2010|C          |20      |6   |
|2010|Ruby       |12      |7   |
|2010|Delphi     |8       |8   |
|2010|AppleScript|3       |9   |
|2010|Bash       |3       |10  |
|2011|PHP        |102     |1   |
|2011|Java       |93      |2   |
|2011|JavaScript |83      |3   |
|2011|Python     |37      |4   |
|2011|Objective-C|34      |5   |
|2011|C          |24      |6   |
|2011|Ruby       |20      |7   |
|2011|Perl       |9       |8   |
|2011|Delphi     |8       |9   |
|2011|Bash       |7       |10  |
|2012|PHP        |154     |1   |
|2012|JavaScript |132     |2   |
|2012|Java       |124     |3   |
|2012|Python     |69      |4   |
|2012|Objective-C|45      |5   |
|2012|C          |27      |6   |
|2012|Ruby       |27      |7   |
|2012|Bash

In [ ]:
Сохраняем итоговый результат в формате Parquet.

In [15]:
(
    top10_by_year
    .write
    .mode("overwrite")
    .parquet(OUTPUT_PATH)
)

Считываем сохраненный результат обратно из Parquet и проверяем, что данные записались корректно.

In [16]:
result = spark.read.parquet(OUTPUT_PATH)
result.orderBy("year", "rank").show(200, truncate=False)

+----+-----------+--------+----+
|year|language   |mentions|rank|
+----+-----------+--------+----+
|2010|Java       |52      |1   |
|2010|PHP        |46      |2   |
|2010|JavaScript |44      |3   |
|2010|Python     |26      |4   |
|2010|Objective-C|23      |5   |
|2010|C          |20      |6   |
|2010|Ruby       |12      |7   |
|2010|Delphi     |8       |8   |
|2010|AppleScript|3       |9   |
|2010|Bash       |3       |10  |
|2011|PHP        |102     |1   |
|2011|Java       |93      |2   |
|2011|JavaScript |83      |3   |
|2011|Python     |37      |4   |
|2011|Objective-C|34      |5   |
|2011|C          |24      |6   |
|2011|Ruby       |20      |7   |
|2011|Perl       |9       |8   |
|2011|Delphi     |8       |9   |
|2011|Bash       |7       |10  |
|2012|PHP        |154     |1   |
|2012|JavaScript |132     |2   |
|2012|Java       |124     |3   |
|2012|Python     |69      |4   |
|2012|Objective-C|45      |5   |
|2012|C          |27      |6   |
|2012|Ruby       |27      |7   |
|2012|Bash

In [17]:
spark.stop()